# 📂 آماده‌سازی داده در Climatology Engine

این نوت‌بوک مراحل آماده‌سازی داده برای پردازش را نشان می‌دهد.

**مواردی که یاد می‌گیرید:**
- بررسی کیفیت داده
- تشخیص داده‌های گمشده
- شناسایی نقاط پرت
- آماده‌سازی داده برای برازش

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
print('✅ کتابخانه‌ها بارگذاری شدند.')

In [ ]:
# بارگذاری داده نمونه
sample_dir = os.path.join(project_root, 'sample_data')
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))

print(f'📊 شکل داده: {station_data.shape}')
station_data.head()

In [ ]:
# بررسی داده‌های گمشده
missing = station_data.isnull().sum()
total = len(station_data)

print("📊 آمار داده‌های گمشده:")
for col in station_data.columns:
    count = missing[col]
    percent = (count / total) * 100
    print(f"   {col}: {count} ({percent:.2f}%)")

if missing.sum() == 0:
    print("\n✅ هیچ داده گمشده‌ای وجود ندارد.")
else:
    print("\n⚠️ داده‌های گمشده وجود دارند.")

In [ ]:
# بررسی داده‌های پرت با روش IQR
data = station_data.values

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, col in enumerate(['tmin', 'tmean', 'tmax']):
    axes[i].boxplot(data[:, i])
    axes[i].set_title(f'Boxplot - {col}')
    axes[i].set_ylabel('دما (°C)')

plt.tight_layout()
plt.show()

In [ ]:
# تشخیص نقاط پرت با روش Z-score
def detect_outliers_zscore(data, threshold=3):
    mean = np.mean(data)
    std = np.std(data)
    z_scores = np.abs((data - mean) / std)
    return z_scores > threshold

outliers = {}
for i, col in enumerate(['tmin', 'tmean', 'tmax']):
    mask = detect_outliers_zscore(data[:, i])
    outliers[col] = np.sum(mask)
    print(f"{col}: {outliers[col]} نقطه پرت")

total_outliers = sum(outliers.values())
print(f"\n✅ تعداد کل نقاط پرت: {total_outliers} ({total_outliers/len(data)*100:.2f}%)")

In [ ]:
# تابع پاک‌سازی داده
def clean_data(data, method='remove', threshold=3):
    """
    پاک‌سازی داده با حذف یا جایگزینی نقاط پرت
    """
    cleaned = data.copy()
    for i in range(data.shape[1]):
        mean = np.mean(data[:, i])
        std = np.std(data[:, i])
        z_scores = np.abs((data[:, i] - mean) / std)
        outlier_mask = z_scores > threshold
        
        if method == 'remove':
            cleaned = cleaned[~outlier_mask]
        elif method == 'replace':
            cleaned[outlier_mask, i] = mean
    
    return cleaned

cleaned_data = clean_data(data, method='remove')
print(f"📊 داده اصلی: {data.shape}")
print(f"📊 داده پاک‌سازی شده: {cleaned_data.shape}")

In [ ]:
# رسم مقایسه داده قبل و بعد از پاک‌سازی
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(['tmean', 'tmax']):
    idx = ['tmin', 'tmean', 'tmax'].index(col)
    axes[i].boxplot([data[:, idx], cleaned_data[:, idx]], labels=['قبل', 'بعد'])
    axes[i].set_title(f'{col} - مقایسه قبل و بعد از پاک‌سازی')
    axes[i].set_ylabel('دما (°C)')

plt.tight_layout()
plt.show()

## 📋 جمع‌بندی

در این نوت‌بوک یاد گرفتید:

✅ بررسی داده‌های گمشده
✅ تشخیص نقاط پرت با Boxplot و Z-score
✅ پاک‌سازی داده‌ها

---

**مراحل بعدی:**
- نوت‌بوک ۰۳: برازش همه توزیع‌ها